## Sales Manager outbound, then reply loop (with proper email threading)

In [ ]:
from __future__ import annotations

import asyncio
import contextvars
import email.utils
import os
import re
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional

import resend
from dotenv import load_dotenv
from fastapi import FastAPI, Request
from openai import AsyncOpenAI
from pydantic import BaseModel, Field

from agents import Agent, Runner, function_tool, set_default_openai_client, set_tracing_export_api_key, trace

# Jupyter cwd is often this folder; walk up so repo-root `.env` (e.g. `agents/.env`) is loaded.
for _d in [Path.cwd(), *Path.cwd().parents]:
    _env = _d / ".env"
    if _env.is_file():
        load_dotenv(_env, override=True)
        break
else:
    load_dotenv(override=True)

_or_key = os.getenv("OPENROUTER_API_KEY")
_oai_key = os.getenv("OPENAI_API_KEY")
if _or_key:
    _client = AsyncOpenAI(api_key=_or_key, base_url="https://openrouter.ai/api/v1")
else:
    _client = AsyncOpenAI(api_key=_oai_key or "")
set_default_openai_client(_client)
if _oai_key:
    set_tracing_export_api_key(_oai_key)

thread_context: contextvars.ContextVar[Optional[Dict[str, Any]]] = contextvars.ContextVar(
    "thread_context", default=None
)

threads: Dict[str, Dict[str, Any]] = {}
msg_id_to_thread: Dict[str, str] = {}

THREAD_TAG_RE = re.compile(r"\[THREAD:([a-f0-9-]{36})\]", re.I)
EMAIL_IN_ANGLE_RE = re.compile(r"<([^>]+@[^>]+)>")


def _parse_from_field(raw: str) -> str:
    _, addr = email.utils.parseaddr(raw or "")
    if addr:
        return addr
    m = EMAIL_IN_ANGLE_RE.search(raw or "")
    return (m.group(1) if m else raw or "").strip()


def _header_value(headers: str, name: str) -> Optional[str]:
    if not headers:
        return None
    for line in headers.replace("\r\n", "\n").split("\n"):
        if ":" in line:
            k, v = line.split(":", 1)
            if k.strip().lower() == name.lower():
                return v.strip()
    return None


def _message_id_token(in_reply_to: Optional[str]) -> Optional[str]:
    if not in_reply_to:
        return None
    m = EMAIL_IN_ANGLE_RE.search(in_reply_to)
    if m:
        return m.group(1).split("@")[0]
    return in_reply_to.strip("<> ").split("@")[0] if in_reply_to else None


def _resolve_thread_id(*, subject: str, headers: str) -> Optional[str]:
    m = THREAD_TAG_RE.search(subject or "")
    if m:
        return m.group(1)
    irt = _header_value(headers or "", "In-Reply-To")
    token = _message_id_token(irt)
    if token and token in msg_id_to_thread:
        return msg_id_to_thread[token]
    if irt:
        clean = irt.strip("<>")
        for mid, tid in list(msg_id_to_thread.items()):
            if mid in clean or clean in mid:
                return tid
    return None


def _headers_for_thread_match(headers: Any) -> str:
    """Resend Receiving API returns `headers` as a dict; `_resolve_thread_id` expects header lines."""
    if isinstance(headers, dict):
        return "\n".join(f"{k}: {v}" for k, v in headers.items())
    return str(headers or "")


def _inbound_user_turn(prefix: str, prospect: str, subject: str, body: str) -> str:
    return f"{prefix}\nFrom: {prospect}\nSubject: {subject}\n\n{body.strip()}\n"


In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send HTML to the prospect. Called by Email Manager after Sales Manager handoff.

    This is the moment the outbound thread becomes real for reply routing: we register
    `threads[thread_id]` here (before the HTTP handler merges full agent history after Runner.run).
    """
    ctx = thread_context.get() or {}
    to_email = ctx.get("prospect_email") or os.environ.get(
        "SALES_PROSPECT_EMAIL", "chrys@wileeilkue.resend.app"
    )
    from_addr = os.environ.get("RESEND_FROM", "Acme <onboarding@resend.dev>")
    reply_to = (os.environ.get("RESEND_REPLY_TO") or "").strip()
    # If the var exists but is empty, getenv(..., default) still returns ""; coalesce to inbox.
    receiving = (os.environ.get("RESEND_RECEIVING_ADDRESS") or "").strip() or (
        "chrys@wileeilkue.resend.app"
    )
    to_norm = to_email.lower()
    if not reply_to and receiving and not to_norm.endswith(".resend.app"):
        reply_to = receiving
    if not reply_to and receiving and "resend.dev" in from_addr.lower():
        reply_to = receiving
    tid = ctx.get("thread_id")
    if tid:
        if tid not in threads:
            threads[tid] = {"history": [], "prospect_email": to_email}
        elif not threads[tid].get("prospect_email"):
            threads[tid]["prospect_email"] = to_email
    if tid and f"[THREAD:{tid}]" not in subject:
        subject = f"{subject} [THREAD:{tid}]"

    api_key = (os.environ.get("RESEND_API_KEY") or "").strip()
    if not api_key:
        return {
            "status": "error",
            "id": "",
            "detail": "RESEND_API_KEY is missing; set it in .env.",
        }
    resend.api_key = api_key
    send_params: Dict[str, Any] = {
        "from": from_addr,
        "to": [to_email],
        "subject": subject,
        "html": html_body,
    }
    if reply_to:
        send_params["reply_to"] = reply_to
    try:
        out = resend.Emails.send(send_params)
    except Exception as e:
        return {
            "status": "error",
            "id": "",
            "detail": str(e),
            "reply_to": reply_to or "",
        }
    mid = None
    if isinstance(out, dict):
        mid = out.get("id")
    elif out is not None:
        mid = getattr(out, "id", None)
    if not mid:
        return {
            "status": "error",
            "id": "",
            "detail": f"Resend returned no message id: {out!r}",
        }
    if tid:
        msg_id_to_thread[str(mid)] = tid
        meta = ctx.setdefault("meta", {})
        meta["last_resend_id"] = str(mid)
    return {"status": "success", "id": str(mid), "reply_to": reply_to or ""}


subject_instructions = (
    "You can write a subject for a sales email. "
    "You are given a message and you need to write a subject that is likely to get a response."
)
html_instructions = (
    "You can convert a text email body to an HTML email body. "
    "You are given a text email body which might have some markdown "
    "and you need to convert it to an HTML email body with simple, clear, compelling layout and design."
)

subject_writer = Agent(
    name="Email subject writer",
    instructions=subject_instructions,
    model="gpt-4o-mini",
)
subject_tool = subject_writer.as_tool(
    tool_name="subject_writer",
    tool_description="Write a subject for a sales email",
)

html_converter = Agent(
    name="HTML email body converter",
    instructions=html_instructions,
    model="gpt-4o-mini",
)
html_tool = html_converter.as_tool(
    tool_name="html_converter",
    tool_description="Convert a text email body to an HTML email body",
)

emailer_tools = [subject_tool, html_tool, send_html_email]
emailer_instructions = (
    "You are an email formatter and sender. You receive the body of an email to be sent. "
    "You first use the subject_writer tool to write a subject for the email, then use the html_converter tool "
    "to convert the body to HTML. Finally, you use the send_html_email tool to send the email with the subject and HTML body."
)

emailer_agent = Agent(
    name="Email Manager",
    instructions=emailer_instructions,
    tools=emailer_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it",
)


In [ ]:
_instructions1 = (
    "You are a sales agent working for ComplAI, "
    "a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. "
    "You write professional, serious cold emails."
)
_instructions2 = (
    "You are a humorous, engaging sales agent working for ComplAI, "
    "a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. "
    "You write witty, engaging cold emails that are likely to get a response."
)
_instructions3 = (
    "You are a busy sales agent working for ComplAI, "
    "a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. "
    "You write concise, to the point cold emails."
)

sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=_instructions1,
    model="gpt-4o-mini",
)
sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=_instructions2,
    model="gpt-4o-mini",
)
sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=_instructions3,
    model="gpt-4o-mini",
)

_description = "Write a cold sales email or a follow-up email body as instructed by the Sales Manager"
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=_description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=_description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=_description)
sales_manager_tools = [tool1, tool2, tool3]
sales_manager_handoffs = [emailer_agent]

sales_manager_instructions = """
You are a Sales Manager at ComplAI. You use the three sales_agent tools and hand off to the Email Manager for HTML and sending.

## Cold outreach (initial outbound)
When the user asks for a cold sales email and the message does NOT start with [INBOUND_REPLY]:
1. Generate drafts: call sales_agent1, sales_agent2, and sales_agent3. Do not proceed until you have received ALL THREE drafts.
2. Evaluate the three drafts and select ONLY the single best one.
3. Hand off EXACTLY ONCE to the Email Manager with ONLY the winning body text. Do not hand off multiple times.

## Prospect reply (after they answered your cold email)
When the user message starts with [INBOUND_REPLY]:
1. Read the prospect's reply and the full conversation history.
2. Decide on the best next response. You may optionally call the sales_agent tools for alternative phrasings.
3. Hand off EXACTLY ONCE to the Email Manager with your chosen reply body text. Do not hand off multiple times.

CRITICAL RULES (follow exactly):
- For cold outreach: you MUST call all three sales agents before evaluating and handing off.
- You must hand off to the Email Manager EXACTLY ONCE per turn — never more than once.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=sales_manager_tools,
    handoffs=sales_manager_handoffs,
    model="gpt-4o-mini",
)


In [ ]:
app = FastAPI(title="ComplAI SDR inbound (chrys)")


class SimulateInbound(BaseModel):
    thread_id: str = Field(
        ...,
        description="UUID from start_cold_campaign() or POST /outbound/cold after the first send.",
    )
    from_email: str = Field(default="prospect@example.com")
    subject: str = Field(default="Question about ComplAI")
    text: str = Field(..., description="Plain-text body of the inbound email")


async def _run_with_context(
    *,
    thread_id: str,
    prospect_email: str,
    agent_input: str | List[Dict[str, Any]],
    trace_name: str = "chrys_sales",
) -> Any:
    ctx: Dict[str, Any] = {"thread_id": thread_id, "prospect_email": prospect_email, "meta": {}}
    tok = thread_context.set(ctx)
    try:
        with trace(trace_name):
            return await Runner.run(sales_manager, agent_input)
    finally:
        thread_context.reset(tok)


@app.get("/health")
async def health():
    return {"ok": True, "threads": len(threads)}


async def start_cold_campaign(brief: str | None = None) -> Dict[str, Any]:
    """Run Sales Manager → Email Manager once. Thread is registered when Email Manager calls send_html_email;
    after the run we store full conversation history for follow-up turns."""
    text = brief or "Send a cold sales email addressed to Dear CEO"
    tid = str(uuid.uuid4())
    prospect = os.environ.get("SALES_PROSPECT_EMAIL", "chrys@wileeilkue.resend.app")
    result = await _run_with_context(
        thread_id=tid,
        prospect_email=prospect,
        agent_input=text,
        trace_name="chrys_outbound_cold",
    )
    threads.setdefault(tid, {"history": [], "prospect_email": prospect})
    threads[tid]["history"] = result.to_input_list()
    threads[tid]["prospect_email"] = prospect
    return {
        "thread_id": tid,
        "final_output": result.final_output,
        "prospect_email": prospect,
    }


@app.post("/outbound/cold")
async def outbound_cold(request: Request):
    """Optional HTTP wrapper: same as `await start_cold_campaign(...)` in the notebook."""
    brief: str | None = None
    try:
        body = await request.json()
        if isinstance(body, dict) and body.get("brief"):
            brief = str(body["brief"])
    except Exception:
        pass
    return await start_cold_campaign(brief)


@app.post("/webhooks/resend")
async def webhook_resend(request: Request):
    """Resend `email.received` JSON webhook; fetch body via Receiving API (same API key as send)."""
    event = await request.json()
    if event.get("type") != "email.received":
        return {"status": "ignored", "reason": event.get("type", "unknown")}

    data = event.get("data") or {}
    email_id = data.get("email_id")
    if not email_id:
        return {"status": "error", "reason": "missing_email_id"}

    resend.api_key = os.environ.get("RESEND_API_KEY", "")
    full = await asyncio.to_thread(
        lambda: resend.Emails.Receiving.get(email_id=email_id)
    )
    if isinstance(full, dict):
        rec = full
    elif hasattr(full, "model_dump"):
        rec = full.model_dump()
    elif hasattr(full, "dict"):
        rec = full.dict()
    else:
        rec = dict(vars(full)) if hasattr(full, "__dict__") else {}

    raw_from = str(rec.get("from") or "")
    subject = str(rec.get("subject") or "")
    text = str(rec.get("text") or rec.get("html") or "")
    headers = _headers_for_thread_match(rec.get("headers"))

    prospect = _parse_from_field(raw_from)
    tid = _resolve_thread_id(subject=subject, headers=headers)
    if not tid or tid not in threads:
        return {
            "status": "ignored",
            "reason": "no_matching_thread",
            "hint": "Run start_cold_campaign() or POST /outbound/cold first. Replies must keep [THREAD:uuid] in the subject or In-Reply-To our outbound Resend message id.",
        }

    record = threads[tid]
    hist: List[Dict[str, Any]] = list(record.get("history") or [])
    user_content = _inbound_user_turn(
        "[INBOUND_REPLY]", prospect, subject, text
    )
    agent_input = hist + [{"role": "user", "content": user_content}]

    result = await _run_with_context(
        thread_id=tid,
        prospect_email=prospect or record.get("prospect_email") or "",
        agent_input=agent_input,
        trace_name="chrys_inbound_reply",
    )
    record["history"] = result.to_input_list()
    record["prospect_email"] = prospect or record.get("prospect_email")

    return {"status": "ok", "thread_id": tid, "final_output": result.final_output}


@app.post("/simulate/inbound")
async def simulate_inbound(payload: SimulateInbound):
    """Local test: fake a prospect reply after start_cold_campaign (same thread_id)."""
    tid = payload.thread_id.strip()
    if tid not in threads:
        return {
            "status": "error",
            "reason": "unknown_thread_id",
            "hint": "Run start_cold_campaign() or POST /outbound/cold first; use the returned thread_id.",
            "thread_id": tid,
        }

    record = threads[tid]
    hist: List[Dict[str, Any]] = list(record.get("history") or [])
    user_content = _inbound_user_turn(
        "[INBOUND_REPLY]", payload.from_email, payload.subject, payload.text
    )
    agent_input = hist + [{"role": "user", "content": user_content}]

    result = await _run_with_context(
        thread_id=tid,
        prospect_email=payload.from_email,
        agent_input=agent_input,
        trace_name="chrys_inbound_reply",
    )
    record["history"] = result.to_input_list()
    record["prospect_email"] = payload.from_email

    return {"status": "ok", "thread_id": tid, "final_output": result.final_output}


In [ ]:
async def debug_sales_manager_flow():
    """Test the Sales Manager flow and see exactly what tools it calls."""
    print("=== Testing Sales Manager flow ===")
    tid = str(uuid.uuid4())
    prospect = os.environ.get("SALES_PROSPECT_EMAIL", "chrys@wileeilkue.resend.app")
    text = "Send a cold sales email addressed to Dear CEO"

    ctx: Dict[str, Any] = {"thread_id": tid, "prospect_email": prospect, "meta": {}}
    tok = thread_context.set(ctx)
    try:
        with trace("debug_sales_manager"):
            result = await Runner.run(sales_manager, text)
            print("Final output:", result.final_output)
            print("Tool calls made:", len(getattr(result, 'messages', [])))
            return result
    finally:
        thread_context.reset(tok)

# Test helper for inbound replies
async def test_inbound_reply(thread_id: str, reply_text: str = "This looks interesting. Can you tell me more about pricing?"):
    """Test the inbound reply flow."""
    if thread_id not in threads:
        print(f"Thread {thread_id} not found. Run start_cold_campaign first.")
        return None

    print(f"=== Testing inbound reply for thread {thread_id} ===")
    prospect = os.environ.get("SALES_PROSPECT_EMAIL", "chrys@wileeilkue.resend.app")
    result = await simulate_inbound(SimulateInbound(
        thread_id=thread_id,
        from_email=prospect,
        subject="Re: Your ComplAI outreach",
        text=reply_text
    ))
    print("Inbound result:", result)
    return result

### Run the HTTP API

**Important:** In Jupyter, **`await server.serve()` blocks** one kernel—you cannot run another cell until you interrupt it. So either: **(1)** run the **cold-outreach** code cell **before** the server cell (same kernel), **(2)** start Uvicorn in a **terminal** and use this notebook (or `curl`) for `start_cold_campaign` / `/outbound/cold`, or **(3)** interrupt the server, run `await start_cold_campaign()`, restart the server.

**Resend UI:** Outbound API sends appear under **Emails** (send logs), not under Receiving. Receiving shows inbound mail to your `*.resend.app` address.

- **From this notebook:** run the cold cell, then the server cell; the server **`await`s** on the Jupyter event loop. **Interrupt** to stop.
- **From a terminal:** `uv run uvicorn assessment:app --host 0.0.0.0 --port 8765` after exporting the notebook to `assessment.py`, or keep a small `serve.py` that calls `uvicorn.run`.

**Try flow (local server is `http`, not `https`):**

**Step 1 — Sales Manager → Email Manager → `send_html_email`:** run the Python cell **below** this section, or:

**Note on current issues:**
- The Sales Manager should now call the 3 sales agents, evaluate them, and hand off **exactly once** to Email Manager (improved instructions).
- If you see 3 emails, the agent is still not following the "exactly one handoff" rule.
- Webhook errors (500) are now caught with better error messages.

```bash
curl -sS -X POST http://127.0.0.1:8765/outbound/cold
```

(Requires the server to be listening if you use `curl`.) Save `thread_id`. Outbound **To** is **`SALES_PROSPECT_EMAIL`**. For replies via **`RESEND_REPLY_TO`** (e.g. `chrys@wileeilkue.resend.app`), see the intro cell; the default loop uses **To** = receiving inbox.

**Step 2 — Simulate a reply (or reply in your Resend inbox to trigger the webhook):**

```bash
curl -sS -X POST http://127.0.0.1:8765/simulate/inbound \
  -H "Content-Type: application/json" \
  -d '{"thread_id":"PASTE_UUID_FROM_STEP_1","from_email":"ceo@company.com","subject":"Re: ...","text":"I want to know more"}'
```


In [ ]:
# Outbound cold email → Resend (check dashboard: Emails). Run this before `await server.serve()` in this kernel,
# or start the server from a terminal and call POST /outbound/cold instead.
out = await start_cold_campaign()
out

In [ ]:
# thread_id = out['thread_id']
# message = await test_inbound_reply(thread_id)
# message

In [ ]:
# Jupyter already has a running asyncio loop; use Server.serve() instead of uvicorn.run().
from uvicorn import Config, Server

config = Config(app, host="0.0.0.0", port=8765, log_level="info")
server = Server(config)
await server.serve()
